In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm

In [ ]:
# Parameters

S = 4800
r = 0.03
contract_multiplier = 100

strikes = np.arange(4400, 5201, 100)
expiries_days = [7, 14, 30, 60]
expiries_years = [t / 365 for t in expiries_days]
expiry_labels = ["7D", "14D", "30D", "60D"]

In [ ]:
# Black-Scholes formula for European call and put options

def black_scholes(S, K, T, r, sigma):

    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    delta_call = norm.cdf(d1)
    delta_put = norm.cdf(d1) - 1

    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))

    vanna = -norm.pdf(d1) * d2 / sigma

    return call_price, put_price, delta_call, delta_put, gamma, vanna


In [ ]:
# Implied Volatility function definition

def get_iv(S, K, base_iv = 0.3):
    
    moneyness = K / S
    
    if moneyness < 1:
        iv = base_iv + 0.15 * (1 - moneyness)
    else:
        iv = base_iv + 0.15 * (moneyness - 1)
    
    return iv

In [ ]:
# IV Surface Construction

strikes_fine = np.linspace(4400, 5200, 50)
expiries_fine = np.linspace(7, 60, 50)
expiries_fine_years = expiries_fine / 365

iv_matrix = np.zeros((len(expiries_fine), len(strikes_fine)))

for i, T in enumerate(expiries_fine_years):
    for j, K in enumerate(strikes_fine):
        iv_matrix[i, j] = get_iv(S, K)

X, Y = np.meshgrid(strikes_fine, expiries_fine)

fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(X, Y, iv_matrix, cmap='RdYlGn_r', edgecolor='none', alpha=0.95)

ax.set_xlabel('Strike', labelpad=10)
ax.set_ylabel('Days to Expiry', labelpad=10)
ax.set_zlabel('Implied Volatility', labelpad=10)
ax.set_title('SX5E Implied Volatility Surface', pad=20)

ax.view_init(elev=25, azim=225)

fig.colorbar(surf, ax=ax, shrink=0.5, label='IV Level')

plt.tight_layout()
plt.show()

In [ ]:
# Build the option book

rows = []

for T, label in zip(expiries_years, expiry_labels):
    for K in strikes:

        sigma = get_iv(S, K)
        call_price, put_price, delta_call, delta_put, gamma, vanna = black_scholes(S, K, T, r, sigma)

        if K < S:
            open_interest = int(np.random.uniform(2000, 8000))
        elif K == S:
            open_interest = int(np.random.uniform(1000, 4000))
        else:
            open_interest = int(np.random.uniform(500, 2000))

        dollar_gamma = gamma * open_interest * contract_multiplier * S ** 2 / 1e8

        rows.append({
    "expiry": label,
    "T": round(T, 4),
    "strike": K,
    "sigma": round(sigma, 4),
    "call_price": round(call_price, 2),
    "put_price": round(put_price, 2),
    "delta_call": round(delta_call, 4),
    "delta_put": round(delta_put, 4),
    "gamma": round(gamma, 6),
    "vanna": round(vanna, 4),
    "open_interest": open_interest,
    "dollar_gamma": round(dollar_gamma, 4)
})

book = pd.DataFrame(rows)
print(book.to_string())

In [ ]:
# GEX Calculation and Visualization

gex = book.groupby("strike")["dollar_gamma"].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))

colors = ["lightskyblue" for x in gex["dollar_gamma"]]

ax.bar(gex["strike"], gex["dollar_gamma"], color=colors, width=60, edgecolor="black")

ax.axhline(y=0, color="black", linewidth=1.2, linestyle="--")
ax.axvline(x=S, color="red", linewidth=1.5, linestyle="--", label=f"Spot: {S}")

ax.set_xlabel("Strike")
ax.set_ylabel("Dollar Gamma (hundreds of millions €)")
ax.set_title("SX5E Net Gamma Exposure by Strike")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Vanna Flow Calculation

vanna_flow_up = []
vanna_flow_down = []

for idx, row in book.iterrows():

    _, _, delta_base, _, _, _ = black_scholes(S, row["strike"], row["T"], r, row["sigma"])
    _, _, delta_up, _, _, _ = black_scholes(S, row["strike"], row["T"], r, row["sigma"] + 0.05)
    _, _, delta_down, _, _, _ = black_scholes(S, row["strike"], row["T"], r, row["sigma"] - 0.05)

    flow_up = (delta_up - delta_base) * row["open_interest"] * contract_multiplier * S / 1e8
    flow_down = (delta_down - delta_base) * row["open_interest"] * contract_multiplier * S / 1e8

    vanna_flow_up.append(flow_up)
    vanna_flow_down.append(flow_down)

book["vanna_flow_up"] = vanna_flow_up
book["vanna_flow_down"] = vanna_flow_down

vanna_up = book.groupby("strike")["vanna_flow_up"].sum().reset_index()
vanna_down = book.groupby("strike")["vanna_flow_down"].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors_up = ["crimson" if x < 0 else "lightskyblue" for x in vanna_up["vanna_flow_up"]]
colors_down = ["crimson" if x < 0 else "lightskyblue" for x in vanna_down["vanna_flow_down"]]

axes[0].bar(vanna_up["strike"], vanna_up["vanna_flow_up"], color=colors_up, width=60, edgecolor="black")
axes[0].axhline(y=0, color="black", linestyle="--")
axes[0].axvline(x=S, color="blue", linestyle="--", label=f"Spot: {S}")
axes[0].set_title("IV Up 5% — Stress")
axes[0].set_xlabel("Strike")
axes[0].set_ylabel("Dealer Flow (hundreds of millions €)")
axes[0].legend()

axes[1].bar(vanna_down["strike"], vanna_down["vanna_flow_down"], color=colors_down, width=60, edgecolor="black")
axes[1].axhline(y=0, color="black", linestyle="--")
axes[1].axvline(x=S, color="blue", linestyle="--", label=f"Spot: {S}")
axes[1].set_title("IV Down 5% — Relief")
axes[1].set_xlabel("Strike")
axes[1].set_ylabel("Dealer Flow (hundreds of millions €)")
axes[1].legend()

plt.suptitle("SX5E Vanna Flow", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Simulation 

np.random.seed(42)

time_steps = 390
spot_path = [4800]
iv_path = [0.20]

gamma_flip = 4700

for t in range(1, time_steps):

    current_spot = spot_path[-1]
    current_iv = iv_path[-1]

    if t < 100:
        iv_change = np.random.normal(0, 0.001)
        base_move = np.random.normal(0, 3)

    elif t < 150:
        iv_change = np.random.normal(0.003, 0.002)
        base_move = np.random.normal(-5, 6)

    elif t < 250:
        iv_change = np.random.normal(0.001, 0.002)
        base_move = np.random.normal(-3, 8)

    elif t < 300:
        iv_change = np.random.normal(-0.004, 0.001)
        base_move = np.random.normal(2, 4)

    else:
        iv_change = np.random.normal(-0.002, 0.001)
        base_move = np.random.normal(3, 4)

    new_iv = max(current_iv + iv_change, 0.10)

    if current_spot < gamma_flip:
        amplifier = 2.0
    else:
        amplifier = 0.5

    vanna_effect = -row["vanna"] * iv_change * 500

    new_spot = current_spot + base_move * amplifier + vanna_effect

    spot_path.append(new_spot)
    iv_path.append(new_iv)

time = list(range(time_steps))

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(time, spot_path, color="royalblue", linewidth=1.5)
axes[0].axhline(y=gamma_flip, color="red", linestyle="--", linewidth=1.2, label=f"Gamma Flip: {gamma_flip}")
axes[0].axhline(y=4800, color="grey", linestyle=":", linewidth=1, label="Opening Spot: 4800")
axes[0].set_title("SX5E Spot Price")
axes[0].set_ylabel("Index Level")
axes[0].legend()

axes[1].plot(time, iv_path, color="crimson", linewidth=1.5)
axes[1].axhline(y=0.20, color="grey", linestyle=":", linewidth=1, label="Base IV: 20%")
axes[1].set_title("Implied Volatility Path")
axes[1].set_ylabel("IV Level")
axes[1].legend()

gex_positive = [x if x > 0 else 0 for x in gex["dollar_gamma"]]
gex_negative = [x if x < 0 else 0 for x in gex["dollar_gamma"]]

axes[2].bar(gex["strike"], gex_positive, color="lightskyblue", width=60, edgecolor="black", label="Positive GEX")
axes[2].bar(gex["strike"], gex_negative, color="crimson", width=60, edgecolor="black", label="Negative GEX")
axes[2].axhline(y=0, color="black", linestyle="--")
axes[2].axvline(x=S, color="blue", linestyle="--", label=f"Opening Spot: {S}")
axes[2].set_title("Dealer GEX by Strike")
axes[2].set_xlabel("Strike")
axes[2].set_ylabel("Dollar Gamma (hundreds of millions €)")
axes[2].legend()

plt.suptitle("SX5E Full Scenario Simulation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()